# Lab W7: Transformer-Mini dari Nol

**Pendamping Bab 06 (Adopsi Repo Riset).**

Tujuan: menulis *scaled dot-product attention* dari nol dengan torch tensor ops, membangun satu Transformer encoder block (attention + FFN + residual + LayerNorm), dan memverifikasinya terhadap implementasi PyTorch (`nn.TransformerEncoderLayer`). Di akhir, train mini pada toy *sequence classification*.

**Prasyarat:** Lab 1c (MLP + backprop), Lab 3b (sequence modeling), Section 2.2 (LayerNorm, GELU).

**Estimasi:** 4-5 jam.
**Pendamping Bab 06 (Adopsi Repo Riset).**

Tujuan: menulis *scaled dot-product attention* dari nol dengan torch tensor ops, membangun satu Transformer encoder block (attention + FFN + residual + LayerNorm), dan memverifikasinya terhadap implementasi PyTorch (`nn.TransformerEncoderLayer`). Di akhir, train mini pada toy *sequence classification*.

**Prasyarat:** Lab 1c (MLP + backprop), Lab 3b (sequence modeling), Section 2.2 (LayerNorm, GELU).

**Estimasi:** 4-5 jam.

## 1. Setup

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Scaled dot-product attention dari nol

Rumus inti (Vaswani et al. 2017): `Attention(Q, K, V) = softmax(QK^T / √d_k) V`.

Tiga hal yang mudah salah saat menulis pertama kali:
- Lupa membagi dengan `√d_k` - softmax jadi terlalu runcing, gradient menyempit.
- Salah transpos `K` (harus `(batch, d_k, seq)` agar `QK^T` menghasilkan `(batch, seq_q, seq_k)`).
- Lupa *masking*: tanpa mask, token masa depan bisa bocor ke prediksi masa lalu dalam setup kausal.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q, K, V: (B, T, d). Return (B, T, d)."""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)   # (B, T, T)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)                     # (B, T, T)
    return attn @ V                                       # (B, T, d)

# Sanity: output shape benar, attention row sum jadi 1
B, T, d = 2, 5, 16
Q = torch.randn(B, T, d); K = torch.randn(B, T, d); V = torch.randn(B, T, d)
out = scaled_dot_product_attention(Q, K, V)
print('out shape:', out.shape)
attn = F.softmax(Q @ K.transpose(-2, -1) / math.sqrt(d), dim=-1)
print('attention row sums (harus 1):', attn.sum(dim=-1)[0].round(decimals=4))

## 3. Single-head attention module

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.WQ = nn.Linear(d_model, d_model, bias=False)
        self.WK = nn.Linear(d_model, d_model, bias=False)
        self.WV = nn.Linear(d_model, d_model, bias=False)
        self.WO = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x, mask=None):
        Q = self.WQ(x); K = self.WK(x); V = self.WV(x)
        out = scaled_dot_product_attention(Q, K, V, mask)
        return self.WO(out)

sha = SingleHeadAttention(16)
y = sha(torch.randn(2, 5, 16))
print('single-head output:', y.shape)

## 4. Transformer encoder block dari nol

Block standar: `x → LN → Attention → + residual → LN → FFN → + residual`. Versi *pre-norm* (LN sebelum sublayer) lebih stabil daripada *post-norm* klasik, dan dipakai kebanyakan implementasi modern.

In [ ]:
class TinyBlock(nn.Module):
    def __init__(self, d_model=16, d_ff=64, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = SingleHeadAttention(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = x + self.drop(self.attn(self.ln1(x), mask))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x

block = TinyBlock(d_model=16)
y = block(torch.randn(2, 5, 16))
print('block output:', y.shape, 'params:', sum(p.numel() for p in block.parameters()))

## 5. Parity check terhadap `nn.TransformerEncoderLayer`

Kita tidak bisa mencocokkan bobot antara dua implementasi (layout parameter multi-head di PyTorch dilebur, pre-norm vs post-norm berbeda). Tapi kita bisa memverifikasi **kelas ekivalensi** - output dua block, diinisialisasi ulang dengan seed yang sama, pada input yang sama, punya bentuk dan skala yang sebanding.

In [ ]:
torch.manual_seed(7)
x = torch.randn(2, 5, 16)
our_block = TinyBlock(d_model=16)
pt_block  = nn.TransformerEncoderLayer(d_model=16, nhead=1, dim_feedforward=64, activation='gelu', batch_first=True, norm_first=True)
y_ours = our_block(x)
y_pt   = pt_block(x)
print('output range ours:', y_ours.min().item(), y_ours.max().item())
print('output range pyt :', y_pt.min().item(),   y_pt.max().item())
print('shape match:', y_ours.shape == y_pt.shape)

## 6. Training toy: sequence classification

Tugas: diberi urutan token acak, tebak apakah total jumlah token di atas ambang tertentu. Tidak berguna di praktik nyata, tapi cukup untuk memastikan block kita belajar sesuatu.

In [ ]:
def make_toy(n=4000, seq_len=16, vocab=20):
    x = torch.randint(0, vocab, (n, seq_len))
    y = (x.sum(dim=1) > (seq_len * (vocab - 1) / 2)).long()
    return x, y

X_tr, y_tr = make_toy(4000); X_va, y_va = make_toy(500)
print('balance train:', y_tr.float().mean().item())

In [ ]:
class ToyClassifier(nn.Module):
    def __init__(self, vocab=20, d=32, seq_len=16):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.randn(1, seq_len, d) * 0.02)
        self.block = TinyBlock(d_model=d, d_ff=64)
        self.head = nn.Linear(d, 2)
    def forward(self, x):
        h = self.embed(x) + self.pos
        h = self.block(h).mean(dim=1)
        return self.head(h)

model = ToyClassifier().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
crit = nn.CrossEntropyLoss()
from torch.utils.data import DataLoader, TensorDataset
dl_tr = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

va_hist = []
for epoch in range(10):
    model.train()
    for Xb, yb in dl_tr:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad()
        crit(model(Xb), yb).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    model.eval()
    with torch.no_grad():
        va_acc = (model(X_va.to(device)).argmax(1) == y_va.to(device)).float().mean().item()
    va_hist.append(va_acc)
    print(f'epoch {epoch+1}: val_acc={va_acc:.4f}')

plt.plot(va_hist, marker='o'); plt.xlabel('epoch'); plt.ylabel('val acc'); plt.title('Toy classification')
plt.show()

## 7. Refleksi

1. Hapus `/ math.sqrt(d_k)` dari attention. Apa yang terjadi pada akurasi validasi di `ToyClassifier`? Apakah Anda bisa menjelaskan efeknya berdasarkan distribusi softmax?
2. Ganti LayerNorm dengan BatchNorm1d pada `TinyBlock`. Training masih stabil? Hubungkan dengan diskusi LayerNorm vs BatchNorm di Section 2.2.
3. Apa yang berubah bila Anda mengganti ke *post-norm* (LN setelah sublayer, bukan sebelum)? Paper Transformer asli memakai post-norm; mengapa komunitas banyak beralih ke pre-norm?